
# Mendeleev clusters

This example demonstrates how to stress test universal machine learning potentials
by simulating a complex nanoparticle containing all the elements that are supported
by the target model. We will use the version 1.5.0 of the PET-MAD universal potential,
which covers 102 elements, using ASE, and [i-PI](https://ipi-code.org).

Clusters are generated by randomly populating a FCC supercell with the supported
elements, and then relaxed. These structures are then used as starting points for
a Replica Exchange Molecular Dynamics (REMD) simulation, where we attempt both
temperature swaps and Monte Carlo atomic swaps to explore the compositional and
structural space of the cluster across a range of temperatures.


First, we import the required libraries.



In [ ]:
from copy import copy
import ase.io
import chemiscope
import matplotlib.pyplot as plt
from metatomic.torch.ase_calculator import MetatomicCalculator
import upet
import numpy as np
import matplotlib.colors as mcolors


# i-PI utilities
from ipi.utils.parsing import read_output, read_trajectory
from ipi.utils.scripting import InteractiveSimulation

First, we download the universal potential model. We will use the extra-small (xs)
version of PET-MAD, version 1.5.0.
The `upet` package provides a handy utility to fetch models and save them locally.



In [ ]:
model_path = "pet-mad-xs-v1.5.0.pt"
upet.save_upet(model="pet-mad", size="xs", version="1.5.0", output=model_path)

## Generating the clusters
We start by generating a collection of random atomic configurations.
The script `make_soup.py` creates FCC supercells, randomly populates them
with the chemical elements supported by the PET-MAD model, and relaxes them.

We have configured `make_soup.py` as a command-line tool. You could invoke it
directly from the command line, passing the path to the model and the number
of clusters to generate.

```bash
python data/make_soup.py --model pet-mad-xs-v1.5.0.pt --n_soup 16
```
In this example, we will use the pre-generated `soup_relaxed-*.xyz` files in the
`data` directory to save time, as geometry optimization of 16 dense
nanoparticles can take a while.
We load these files, enlarge the simulation cell to 40x40x40 Å, and shift the
atoms to the center of the cell. The resulting files are saved in the root folder.



In [ ]:
for i in range(16):
    atoms = ase.io.read(f"data/soup_relaxed-{i:02d}.xyz")
    atoms.set_cell([40.0, 40.0, 40.0])
    atoms.center()
    ase.io.write(f"nano_soup-{i:02d}.xyz", atoms)

# Let's load and visualize the starting configurations.

initial_structures = [ase.io.read(f"nano_soup-{i:02d}.xyz") for i in range(16)]
chemiscope.show(
    initial_structures,
    mode="structure",
    settings=chemiscope.quick_settings(
        structure_settings={
            "unitCell": True,
            "bonds": False,
            "spaceFilling": True,
            "keepOrientation": True,
        },
    ),
)

## Running Replica Exchange Molecular Dynamics
We use `i-PI` to run a REMD simulation. The input file `input-remd.xml`
defines 16 replicas spanning a temperature range from 300 K to 3000 K.

A ``<smotion mode='remd'>`` block attempts exchanges between adjacent replicas.
Furthermore, a ``<motion mode='atomswap'>`` block inside the standard MD
integrator allows Monte Carlo swapping of chemical identities during
the dynamics.

Let's inspect the `i-PI` input XML. Note the definition of the 16 replicas
using a ``<system_template>`` block, and the configuration of the REMD and
atomic swap moves. You may also note the thermostatting setup that combines
a stochastic velocity rescaling thermostat (that samples quickly changes in
total kinetic energy without slowing down diffusion) with an optimal-sampling
GLE thermostat, that improves sampling of local fluctuations. Small correlation
time of the energy is essential to quickly equilibrate the system, and maximize
the acceptance of the REMD swaps.



In [ ]:
with open("data/input-remd.xml", "r") as f:
    xml_input = f.read()
print(xml_input)

We will use ``InteractiveSimulation`` to run i-PI directly from this Python script.

<div class="alert alert-danger"><h4>Warning</h4><p>Running 16 replicas of a 108-atom system using a ML potential
    is computationally intensive. Here we run only 100 steps for demonstration.</p></div>

Look at `data/input-remd_production.xml` for a more realistic production setup,
which uses a more accurate "S" sized model, a more conservative time step,
and runs for 100 ps. You can run it from the command line with:

```bash
i-pi input-remd_production.xml
```


In [ ]:
sim = InteractiveSimulation(xml_input)

# Run a short simulation (still these are > 4000 force evaluations,
# so it may take a while).
sim.run(250)

## Analyzing the Results
Once the simulation finishes, `i-PI` produces trajectory files and output
logs for each replica.

Let's load the trajectories and output data of all 16 replicas.



In [ ]:
trajectories = []
data_list = []
for i in range(16):
    data, info = read_output(f"{i:02d}_soup-nano.out")
    trj = read_trajectory(f"{i:02d}_soup-nano.pos_0.extxyz", format="ase")
    trajectories.append(trj)
    data_list.append(data)

We can now visualize the potential energy of all the replicas over time.
The thin lines follow the continuous trajectory of each replica, while
the points are colored by the current target temperature.
You can see the initial swaps that re-order replicas matching temperature
and potential energy, followed by slow relaxation of the structures and
a few additional swaps.



In [ ]:
cmap_lines = plt.get_cmap("tab20")
cmap_points = plt.get_cmap("coolwarm")
norm = mcolors.LogNorm(vmin=300, vmax=3000)

fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for i in range(16):
    # Thin lines showing the continuous trajectory of a physical replica
    ax.plot(
        data_list[i]["time"],
        data_list[i]["potential"],
        "-",
        color=cmap_lines(i),
        linewidth=0.8,
        alpha=0.7,
    )
    # Points colored by the current temperature of that replica
    ax.scatter(
        data_list[i]["time"],
        data_list[i]["potential"],
        c=data_list[i]["ensemble_temperature"],
        cmap=cmap_points,
        norm=norm,
        s=15,
        zorder=3,
    )

ax.set_xlabel("Time (ps)")
ax.set_ylabel("Potential Energy (eV)")

# Add a colorbar for the temperature
sm = plt.cm.ScalarMappable(cmap=cmap_points, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax)
cbar.set_label("Target Temperature (K)")

plt.show()

To visualize the REMD trajectory, combine the 16 replicas into a single
trajectory. Each frame of this combined trajectory contain all
16 clusters, arranged to form a 4x4 grid.
We also use chemiscope's `shapes` to put wireframe boxes around each
cluster, colored according to the temperature of the replica at that step.



In [ ]:
n_frames = min(len(t) for t in trajectories)
combined_trj = []
temperatures_over_time = {i: [] for i in range(16)}

# The trajectory stride is 10 and property stride is 2.
# Therefore, frame f corresponds to data index f * (10 // 2) = f * 5.
for f in range(n_frames):
    data_idx = f * 5
    if data_idx >= len(data_list[0]["step"]):
        data_idx = len(data_list[0]["step"]) - 1

    cell = [160.0, 160.0, 40.0]
    merged_atoms = ase.Atoms(cell=cell, pbc=True)

    for i in range(16):
        row = i // 4
        col = i % 4

        atoms = trajectories[i][f].copy()
        # Shift the atoms so they form a 4x4 grid.
        # Original cell is 40x40x40.
        shift = np.array([col * 40.0, row * 40.0, 0.0])
        atoms.positions += shift
        merged_atoms += atoms

        T = data_list[i]["ensemble_temperature"][data_idx]
        temperatures_over_time[i].append(T)

    combined_trj.append(merged_atoms)

# Create shapes for each cluster
shapes = {}
cmap = plt.get_cmap("coolwarm")
norm = mcolors.LogNorm(vmin=300, vmax=3000)

L = 38.0
edges = [
    ([-L / 2, -L / 2, -L / 2], [L, 0, 0]),
    ([-L / 2, -L / 2, -L / 2], [0, L, 0]),
    ([-L / 2, -L / 2, -L / 2], [0, 0, L]),
    ([L / 2, -L / 2, -L / 2], [0, L, 0]),
    ([L / 2, -L / 2, -L / 2], [0, 0, L]),
    ([-L / 2, L / 2, -L / 2], [L, 0, 0]),
    ([-L / 2, L / 2, -L / 2], [0, 0, L]),
    ([-L / 2, -L / 2, L / 2], [L, 0, 0]),
    ([-L / 2, -L / 2, L / 2], [0, L, 0]),
    ([L / 2, L / 2, -L / 2], [0, 0, L]),
    ([L / 2, -L / 2, L / 2], [0, L, 0]),
    ([-L / 2, L / 2, L / 2], [L, 0, 0]),
]

for i in range(16):
    row = i // 4
    col = i % 4

    # Pre-calculate colors for this cluster over all frames
    colors = []
    for f in range(n_frames):
        T = temperatures_over_time[i][f]
        rgba = cmap(norm(T))
        # 3dmol.js has limited support for alpha in shapes, so we drop it
        hex_color = mcolors.to_hex(rgba, keep_alpha=False).upper().replace("#", "0x")
        colors.append(hex_color)

    for edge_idx, (start, vector) in enumerate(edges):
        structure_params = []
        for f in range(n_frames):
            cx = col * 40.0 + 20.0
            cy = row * 40.0 + 20.0
            cz = 20.0

            structure_params.append(
                {
                    "position": [cx + start[0], cy + start[1], cz + start[2]],
                    "color": colors[f],
                }
            )

        shapes[f"cluster_{i}_edge_{edge_idx}"] = {
            "kind": "cylinder",
            "parameters": {
                "global": {"radius": 1.0, "vector": vector},
                "structure": structure_params,
            },
        }

Finally, we can use `chemiscope` to visualize the combined trajectory.



In [ ]:
chemiscope.show(
    structures=combined_trj,
    mode="structure",
    shapes=shapes,
    settings=chemiscope.quick_settings(
        structure_settings={
            "unitCell": False,
            "bonds": False,
            "spaceFilling": True,
            "keepOrientation": True,
            "shape": list(shapes.keys()),
        },
    ),
)

## Quantitative validation

Stability of the simulation, and qualitative behavior (e.g. what atoms move to
the surface and the gas phase at different temperatures) can be visually inspected
from the trajectory. In order to obtain more quantitative validation, one can
compare the forces computed by the model to reference DFT calculations on a
subset of the structures.

Here we use a handful of frames obtained from the end of a 100ps-long REMD simulation,
recomputed with FHI-aims. Note that converging DFT calculations on these large, highly
unusual structures is challenging, and required slight modifications of the settings
relative to those used for reference MAD-1.5 calculations, such as tighter integration
grids, or using open boundary conditions.
This changes the reference energies, but has minimal effect on the forces.



In [ ]:
frames_aims = []
frames_petmad = []
r2scan_avail = {0: "01", 3: "04", 7: "08", 9: "10", 4: "11"}
calc = MetatomicCalculator(model=model_path)

for i in r2scan_avail:
    frame = ase.io.read(f"data/structure_{r2scan_avail[i]}.xyz")
    frames_aims.append(frame)

    frame = frame.copy()
    frame.pbc = False
    frame.calc = copy(calc)
    frame.get_forces()
    frames_petmad.append(frame)

We visualize the structures and forces with chemiscope.
Note that errors are larger than what would be obtained using the more accurate
"S" sized model, but they already indicate good semi-quantitative agreement despite
the challenging nature of the test.



In [ ]:
f_petmad = np.vstack([f.get_forces() for f in frames_petmad])
f_aims = np.vstack([f.get_forces() for f in frames_aims])

vec_petmad = chemiscope.ase_vectors_to_arrows(frames_petmad, "forces", scale=1)
vec_aims = chemiscope.ase_vectors_to_arrows(frames_aims, "forces", scale=1)
vec_petmad["parameters"]["global"]["color"] = "red"
vec_aims["parameters"]["global"]["color"] = "black"

chemiscope.show(
    frames_petmad,
    properties={
        "aims-x": f_aims[:, 0],
        "aims-y": f_aims[:, 1],
        "aims-z": f_aims[:, 2],
        "petmad-x": f_petmad[:, 0],
        "petmad-y": f_petmad[:, 1],
        "petmad-z": f_petmad[:, 2],
    },
    shapes={
        "forces-aims": vec_aims,
        "forces-petmad": vec_petmad,
    },
    environments=chemiscope.all_atomic_environments(frames_petmad),
    settings=chemiscope.quick_settings(
        x="aims-x",
        y="petmad-x",
        periodic=True,
        target="atom",
        structure_settings={
            "environments": {"activated": False},
            "keepOrientation": True,
            "bonds": False,
            "supercell": {"0": 1, "1": 1, "2": 1},
            "shape": ["forces-aims", "forces-petmad"],
        },
    ),
)